# SSL400 EXP5: EfficientNetV2-S + BiLSTM + CLAHE
## Full standalone package - just upload raw videos and run!
---
### Before running:
1. Upload your raw videos to this Drive account:
   - `My Drive / SSL400_EXP5 / data / raw / `
   - Inside `raw/` put your 8 class folders (Thank_you, Hello, Good, House, Eat, Drink, Tell, Write)
2. Extract this zip to: `My Drive / SSL400_EXP5 /`
3. Run cells 1 to 8 in order

| Cell | What it does | Time |
|---|---|---|
| 1 | Mount Drive + setup | 1 min |
| 2 | Install packages | 2 min |
| 3 | Generate train/val/test splits | 1 min |
| 4 | Process videos to .npy with CLAHE | 30-60 min |
| 5 | Build EfficientNetV2 + BiLSTM model | 1 min |
| 6 | Load dataset | 1 min |
| 7 | Train Phase 1 + Phase 2 | 4-5 hours |
| 8 | Evaluate + graphs | 5 min |

In [ ]:
# Cell 1 - MOUNT GOOGLE DRIVE & SETUP
from google.colab import drive
drive.mount('/content/drive')
import os, shutil

# =====================================================
# UPDATE THIS IF YOUR FOLDER NAME IS DIFFERENT
PACKAGE_DIR  = '/content/drive/MyDrive/SSL400_EXP5'
DRIVE_SAVE   = '/content/drive/MyDrive/SSL400_EXP5/models/experiment_5'
# =====================================================

LOCAL = '/content/ssl400'
os.makedirs(LOCAL, exist_ok=True)

# Copy EVERYTHING (code + raw data + splits) to fast local SSD
print('Copying code and ALL data to local SSD (this might take a minute)...')
if os.path.exists(LOCAL):
    shutil.rmtree(LOCAL)
shutil.copytree(PACKAGE_DIR, LOCAL)

os.makedirs(DRIVE_SAVE, exist_ok=True)
os.chdir(LOCAL)

# Check raw videos
raw_dir = os.path.join(LOCAL, 'data', 'raw')
classes = os.listdir(raw_dir) if os.path.exists(raw_dir) else []
print('\nWorking directory:', os.getcwd())
print('Raw video classes found:', classes)
if not classes:
    print('WARNING: No raw videos found inside data/raw!')
else:
    total = sum(len(os.listdir(os.path.join(raw_dir, c))) 
                for c in classes if os.path.isdir(os.path.join(raw_dir, c)))
    print('Total video files ready for processing:', total)
    print('Setup complete!')


In [ ]:
# Cell 2 - INSTALL DEPENDENCIES
!pip install tf-keras tf-models-official ultralytics --quiet
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'
print('All dependencies installed!')

In [ ]:
# Cell 3 - GENERATE TRAIN/VAL/TEST SPLITS
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'

splits_done = all([
    os.path.exists('/content/ssl400/data/splits/train_split.csv'),
    os.path.exists('/content/ssl400/data/splits/val_split.csv'),
    os.path.exists('/content/ssl400/data/splits/test_split.csv')
])

if splits_done:
    import csv
    with open('/content/ssl400/data/splits/train_split.csv') as f:
        n = sum(1 for _ in csv.DictReader(f))
    print('Splits already exist! Train samples:', n)
else:
    print('Generating splits from raw videos...')
    !python /content/ssl400/src/data/generate_splits.py
    print('Splits generated!')


In [ ]:
# Cell 4 - PROCESS RAW VIDEOS -> .NPY FRAMES
# EXP5 uses CLAHE + Gamma (same as EXP2) with 32 frames
# This takes 30-60 minutes - DO NOT disconnect!
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'

proc_dir  = '/content/ssl400/data/processed/exp5_clahe_gamma'
drive_npy = '/content/drive/MyDrive/SSL400_EXP5/data/processed/exp5_clahe_gamma'

# Check if already processed (resume support)
if os.path.exists(proc_dir):
    npy_count = len([f for f in os.listdir(proc_dir) if f.endswith('.npy')])
else:
    npy_count = 0

if npy_count > 400:
    print('Already processed! Found', npy_count, '.npy files. Skipping.')
else:
    print('Processing raw videos with CLAHE + Gamma (32 frames)...')
    print('This takes 30-60 minutes. Do NOT disconnect!')
    !python /content/ssl400/src/data/video_to_frames.py --exp_id 5

    npy_count = len([f for f in os.listdir(proc_dir) if f.endswith('.npy')]) if os.path.exists(proc_dir) else 0
    print('Done! Created', npy_count, '.npy files')

    # Backup to Drive so you never need to re-process!
    import shutil
    print('Backing up .npy files to Drive...')
    os.makedirs(drive_npy, exist_ok=True)
    for fn in os.listdir(proc_dir):
        if fn.endswith('.npy'):
            shutil.copy2(os.path.join(proc_dir, fn), os.path.join(drive_npy, fn))
    print('Backup complete! Next time this cell will be skipped automatically.')


In [ ]:
# Cell 5 - BUILD EfficientNetV2-S + BiLSTM MODEL
import tensorflow as tf, numpy as np, yaml, os
try:
    import tf_keras as keras
except ImportError:
    keras = tf.keras

with open('/content/ssl400/config.yaml') as f:
    config = yaml.safe_load(f)

NUM_FRAMES  = config['frames']['num_frames']
IMG_H       = config['frames']['height']
IMG_W       = config['frames']['width']
NUM_CLASSES = config['dataset']['num_classes']
SEED        = config['project']['seed']
BATCH_SIZE  = 2
PROC_DIR    = '/content/ssl400/data/processed/exp5_clahe_gamma'
TRAIN_CSV   = '/content/ssl400/data/splits/train_split.csv'
VAL_CSV     = '/content/ssl400/data/splits/val_split.csv'
TEST_CSV    = '/content/ssl400/data/splits/test_split.csv'
LOCAL_MDL   = '/content/exp5_model'
LOG_PATH    = '/content/exp5_log.csv'
os.makedirs(LOCAL_MDL, exist_ok=True)
tf.random.set_seed(SEED)
np.random.seed(SEED)

print('Frames:', NUM_FRAMES, '| Classes:', NUM_CLASSES, '| Batch:', BATCH_SIZE)

backbone = tf.keras.applications.EfficientNetV2S(
    include_top=False, weights='imagenet',
    pooling='avg', input_shape=(IMG_H, IMG_W, 3)
)
backbone.trainable = False

inp = keras.Input(shape=(NUM_FRAMES, IMG_H, IMG_W, 3))
x   = keras.layers.TimeDistributed(backbone, name='efficientnetv2')(inp)
x   = keras.layers.TimeDistributed(keras.layers.BatchNormalization())(x)
x   = keras.layers.Bidirectional(keras.layers.LSTM(256), name='bilstm')(x)
x   = keras.layers.Dropout(0.4)(x)
out = keras.layers.Dense(NUM_CLASSES, activation='softmax')(x)
model = keras.Model(inputs=inp, outputs=out)
print('Model parameters:', model.count_params())
model.summary()


In [ ]:
# Cell 6 - LOAD DATASET
import csv, numpy as np, tensorflow as tf

def load_ds(csv_path, batch_size):
    samples = []
    with open(csv_path) as f:
        reader = csv.DictReader(f)
        cols = reader.fieldnames
        lc = 'label' if 'label' in cols else cols[-1]
        for row in reader:
            stem = os.path.splitext(row[cols[0]])[0]
            p = os.path.join(PROC_DIR, stem + '.npy')
            if os.path.exists(p):
                samples.append((p, int(row[lc])))
    print(' ', os.path.basename(csv_path), ':', len(samples), 'samples')

    def gen():
        for path, label in samples:
            frames = np.load(path).astype(np.float32)
            if frames.shape[0] != NUM_FRAMES:
                idx = np.linspace(0, frames.shape[0]-1, NUM_FRAMES, dtype=int)
                frames = frames[idx]
            frames = np.clip(frames * 127.5 + 127.5, 0, 255)
            oh = np.zeros(NUM_CLASSES, np.float32)
            oh[label] = 1.0
            yield frames, oh

    ds = tf.data.Dataset.from_generator(gen, output_signature=(
        tf.TensorSpec((NUM_FRAMES, IMG_H, IMG_W, 3), tf.float32),
        tf.TensorSpec((NUM_CLASSES,), tf.float32)
    )).batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds, len(samples)

print('Loading dataset...')
train_ds, ntr = load_ds(TRAIN_CSV, BATCH_SIZE)
val_ds,   nvl = load_ds(VAL_CSV,   BATCH_SIZE)
test_ds,  nts = load_ds(TEST_CSV,  BATCH_SIZE)
print('Train:', ntr, '| Val:', nvl, '| Test:', nts)
print('Ready to train!')


In [ ]:
# Cell 7 - TRAIN (Phase 1 frozen backbone -> Phase 2 full fine-tune)
import shutil, os

DRIVE_SAVE = '/content/drive/MyDrive/SSL400_EXP5/models/experiment_5'

def sync(phase):
    os.makedirs(DRIVE_SAVE, exist_ok=True)
    for fn in os.listdir(LOCAL_MDL):
        shutil.copy2(os.path.join(LOCAL_MDL, fn), os.path.join(DRIVE_SAVE, fn))
    if os.path.exists(LOG_PATH):
        shutil.copy2(LOG_PATH, os.path.join(DRIVE_SAVE, 'training_log_' + phase + '.csv'))
    print('  Saved to Drive!')

# Phase 1: Frozen backbone
print('PHASE 1: Frozen EfficientNetV2-S backbone')
model.compile(
    optimizer=keras.optimizers.Adam(0.001, clipnorm=1.0),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy']
)
model.fit(train_ds, validation_data=val_ds, epochs=30, callbacks=[
    keras.callbacks.EarlyStopping('val_loss', patience=15, restore_best_weights=True, verbose=1),
    keras.callbacks.ModelCheckpoint(
        LOCAL_MDL + '/best_model_phase1.keras',
        monitor='val_loss', save_best_only=True, verbose=1
    ),
    keras.callbacks.CSVLogger(LOG_PATH, append=False),
    keras.callbacks.ReduceLROnPlateau('val_loss', factor=0.5, patience=5, verbose=1)
])
sync('phase1')

# Phase 2: Full fine-tune
print('PHASE 2: Full fine-tuning - all layers unfrozen')
for layer in model.layers:
    layer.trainable = True
model.compile(
    optimizer=keras.optimizers.Adam(0.0001, clipnorm=1.0),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy']
)
model.fit(train_ds, validation_data=val_ds, epochs=50, callbacks=[
    keras.callbacks.EarlyStopping('val_loss', patience=25, restore_best_weights=True, verbose=1),
    keras.callbacks.ModelCheckpoint(
        LOCAL_MDL + '/best_model_phase2.keras',
        monitor='val_loss', save_best_only=True, verbose=1
    ),
    keras.callbacks.CSVLogger(LOG_PATH, append=False),
    keras.callbacks.ReduceLROnPlateau('val_loss', factor=0.5, patience=8, verbose=1),
    keras.callbacks.LambdaCallback(
        on_epoch_end=lambda e, l: sync('phase2') if (e + 1) % 5 == 0 else None
    )
])
sync('phase2')
print('Training complete! Model saved to Drive.')


In [ ]:
# Cell 8 - EVALUATE + PER-CLASS REPORT + GRAPHS
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.metrics import classification_report, confusion_matrix
import json, numpy as np, matplotlib.pyplot as plt, seaborn as sns, pandas as pd, shutil

CLASS_NAMES = ['Thank you','Hello','Good','House','Eat','Drink','Tell','Write']

print('Evaluating on test set...')
logits_all, labels_all = [], []
for xb, yb in test_ds:
    logits_all.append(model.predict(xb, verbose=0))
    labels_all.append(yb.numpy())

logits = np.concatenate(logits_all)
y_true = np.argmax(np.concatenate(labels_all), axis=1)
y_pred = np.argmax(logits, axis=1)

top1 = accuracy_score(y_true, y_pred)
top5 = sum(1 for t, l in zip(y_true, logits) if t in np.argsort(l)[-5:]) / len(y_true)
f1   = f1_score(y_true, y_pred, average='macro', zero_division=0)
prec = precision_score(y_true, y_pred, average='macro', zero_division=0)
rec  = recall_score(y_true, y_pred, average='macro', zero_division=0)
rpt_dict = classification_report(y_true, y_pred, target_names=CLASS_NAMES, output_dict=True, zero_division=0)
rpt_str  = classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0)
cm = confusion_matrix(y_true, y_pred)

print('EXP5: EfficientNetV2-S + BiLSTM + CLAHE')
print('Top-1 Accuracy:', round(top1 * 100, 2), '%')
print('Top-5 Accuracy:', round(top5 * 100, 2), '%')
print('Macro F1      :', round(f1, 4))
print('vs EXP2       : 71.43%')
print('Improvement   :', round((top1 - 0.7143) * 100, 2), 'pp')
print(rpt_str)

# Save results
results = {'exp_id': 5, 'exp_name': 'EfficientNetV2-S + BiLSTM + CLAHE',
           'top1_accuracy': float(top1), 'top5_accuracy': float(top5),
           'macro_f1': float(f1), 'macro_precision': float(prec), 'macro_recall': float(rec),
           'classification_report': rpt_dict}
with open('/content/exp5_metrics.json', 'w') as f:
    json.dump(results, f, indent=2)
np.save('/content/exp5_cm.npy', cm)
shutil.copy2('/content/exp5_metrics.json', os.path.join(DRIVE_SAVE, 'exp5_metrics.json'))
shutil.copy2('/content/exp5_cm.npy',       os.path.join(DRIVE_SAVE, 'exp5_cm.npy'))

# Graphs
fig, axes = plt.subplots(1, 3, figsize=(22, 6))
fig.suptitle('EXP5: EfficientNetV2-S + BiLSTM + CLAHE  |  Top-1: ' + str(round(top1*100, 2)) + '%',
             fontsize=14, fontweight='bold')

if os.path.exists(LOG_PATH):
    df = pd.read_csv(LOG_PATH)
    axes[0].plot(df['accuracy'],     label='Train', color='#2ca02c', lw=2)
    axes[0].plot(df['val_accuracy'], label='Val',   color='#d62728', lw=2, ls='--')
    axes[0].set_title('Accuracy Curve', fontsize=13, fontweight='bold')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy'); axes[0].legend()

f1s  = [rpt_dict[c]['f1-score'] * 100 for c in CLASS_NAMES]
clrs = ['#2ecc71' if v >= 70 else '#f39c12' if v >= 50 else '#e74c3c' for v in f1s]
bars = axes[1].bar(CLASS_NAMES, f1s, color=clrs, edgecolor='black', lw=0.5)
axes[1].axhline(top1 * 100, color='blue', ls='--', lw=1.5, label='Overall')
axes[1].set_title('Per-Class F1-Score', fontsize=13, fontweight='bold')
axes[1].set_ylabel('F1 (%)'); axes[1].set_ylim(0, 115)
axes[1].tick_params(axis='x', rotation=45); axes[1].legend()
for b, v in zip(bars, f1s):
    axes[1].text(b.get_x() + b.get_width()/2, b.get_height() + 1,
                 str(round(v, 1)) + '%', ha='center', va='bottom', fontsize=9, fontweight='bold')

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[2],
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
axes[2].set_title('Confusion Matrix', fontsize=13, fontweight='bold')
axes[2].set_xlabel('Predicted'); axes[2].set_ylabel('True')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('/content/exp5_results.png', dpi=150, bbox_inches='tight')
shutil.copy2('/content/exp5_results.png', os.path.join(DRIVE_SAVE, 'exp5_results.png'))
plt.show()
print('All results and graphs saved to Drive!')
